# 01 — Continued Pretraining + Supervised Fine-Tuning (LoRA/QLoRA, Unsloth)
### Building "PostTraining Tutor" — Stage 1 & Stage 2

This notebook takes a general-purpose instruct model and:
1. **Stage 1 — Continued Pretraining**: adapts it to LLM/post-training domain vocabulary using raw text (`llm_post_training_corpus.txt`).
2. **Stage 2 — Supervised Fine-Tuning (SFT)**: teaches it to answer instructions well using `instruction_dataset.jsonl`, trained with LoRA + QLoRA via Unsloth.

Run top to bottom on a Colab GPU runtime (`Runtime > Change runtime type > T4 GPU` is enough).


## Step 0 — Install dependencies

**What:** installing Unsloth (fast LoRA/QLoRA training) plus the standard HF stack.
**Why:** Unsloth patches Transformers/PEFT internals for 2-5x faster training and much lower VRAM use - this is what makes fine-tuning feasible on a free Colab GPU.
**How:** Colab already has PyTorch + CUDA; we only add the libraries on top.

In [ ]:
# WHAT: install Unsloth and the training stack Unsloth depends on.
# WHY: Unsloth is not preinstalled on Colab, and we pin trl/peft/transformers so
#      the training APIs below (SFTTrainer, LoraConfig) match what this notebook expects.
# HOW: pip installs run once per Colab session; safe to re-run if the runtime resets.
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes transformers datasets


## Step 1 — Import libraries

**What:** bring in Unsloth's `FastLanguageModel` (a drop-in, optimized wrapper around HF `AutoModelForCausalLM`), PEFT/TRL trainers, and `datasets`.
**Why:** each library owns one job - Unsloth loads/optimizes the model, PEFT manages LoRA adapters, TRL's `SFTTrainer` handles the instruction-tuning loop, `datasets` handles data loading.
**How:** standard imports; no custom training loop needed since `SFTTrainer` implements it for us.

In [ ]:
# WHAT: core imports for model loading, LoRA config, and the SFT training loop.
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
import json


## Step 2 — Load the base model (and why this model)

**Model choice: `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`**

- **Instruct-tuned already**: it can already follow a chat format, so our SFT stage is *refining* behavior, not teaching instruction-following from zero - realistic for a short demo.
- **3B parameters**: small enough to fully fit and train on a free Colab T4 (16GB VRAM) with 4-bit quantization, but large enough to produce coherent, non-trivial explanations.
- **Pre-quantized (`bnb-4bit`)**: Unsloth ships a bitsandbytes 4-bit version so we skip a slow on-the-fly quantization step at load time.
- **Unsloth-optimized**: this exact checkpoint is one of Unsloth's officially supported models, so `FastLanguageModel` applies its fused-kernel speedups without extra configuration.

**WHAT we're doing below:** loading the 4-bit base model + tokenizer.
**WHY:** 4-bit loading (`load_in_4bit=True`) is what makes QLoRA possible - the *frozen* base weights sit in 4-bit precision, and only the small LoRA adapter weights we add next are trained in higher precision.
**HOW:** `FastLanguageModel.from_pretrained` mirrors `AutoModelForCausalLM.from_pretrained` but returns an Unsloth-patched model.

In [ ]:
# WHAT: load the quantized base model + its tokenizer.
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048   # WHAT: max tokens per training example.
                        # WHY: long enough for instruction+explanation pairs, short enough to keep memory low.
DTYPE = None            # WHAT: let Unsloth auto-pick the best dtype (bf16 on modern GPUs, fp16 on T4).
LOAD_IN_4BIT = True     # WHAT: enables QLoRA - base weights stay 4-bit and frozen.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)


## Step 3 — Configure LoRA (and QLoRA)

**What LoRA does:** instead of updating all ~3B parameters, LoRA freezes the base model and injects small trainable low-rank matrices (A and B) into selected layers. Training only these matrices means training <1% of the total parameters.

**What QLoRA adds on top:** the frozen base weights are kept in 4-bit precision (done above via `load_in_4bit=True`) while the LoRA adapter matrices themselves are trained in higher precision (bf16/fp16). This is *why* QLoRA lets a 3B (or even 70B) model train on a single consumer GPU - the memory-heavy frozen weights are compressed, and only the small trainable part needs full precision.

**Key parameters explained:**
- `r` (**rank**, e.g. 16): the size of the low-rank bottleneck. Higher `r` -> more trainable capacity -> better fit but more memory and higher overfitting risk on small datasets. 8-32 is a common range for small SFT datasets.
- `lora_alpha` (e.g. 16): a scaling factor for the LoRA update (`alpha / r` scales how much the adapter output affects the model). A common convention is `alpha == r` or `alpha == 2*r`.
- `lora_dropout` (e.g. 0.05): dropout applied to the LoRA path during training, as a light regularizer against overfitting on small instruction sets. `0` is also common and slightly faster.
- `target_modules`: which weight matrices get LoRA adapters. We target the attention and MLP projection layers (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`) - this is the standard, well-tested set for Llama-family models.
- `use_gradient_checkpointing="unsloth"`: trades a bit of compute for a large memory saving by not storing all intermediate activations - Unsloth's version is optimized further than the default HF implementation.

In [ ]:
# WHAT: wrap the base model with trainable LoRA adapters (this IS the QLoRA setup,
#       since the base model underneath is already 4-bit from Step 2).
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                       # LoRA rank - trainable bottleneck size.
    lora_alpha=16,               # scaling factor, paired with r above.
    lora_dropout=0.05,           # light regularization for a small dataset.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",                 # WHY "none": training bias terms rarely helps LoRA and adds params.
    use_gradient_checkpointing="unsloth",  # memory saver - lets us use a larger batch/seq length.
    random_state=42,
)

# WHAT: quick sanity check - how many parameters are actually trainable now?
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")


## Step 4 — Stage 1: Continued Pretraining on the domain corpus

**What:** a short additional pretraining pass over `llm_post_training_corpus.txt` using plain causal language modeling (predict-the-next-token), *not* an instruction format.

**Why:** this nudges the model's internal representations toward our domain (Transformers/LoRA/DPO/RLHF terminology and phrasing) before we teach it to answer instructions about that domain - giving Stage 2 (SFT) a better starting point.

**How:** we chunk the raw text into `MAX_SEQ_LENGTH`-sized blocks and reuse the same `SFTTrainer` in "text-only" mode (no chat template, no completion masking) - this is a common practical shortcut for a short domain-adaptation pass without writing a separate raw-LM training loop.

In [ ]:
# WHAT: load and chunk the raw domain corpus for continued pretraining.
with open("data/llm_post_training_corpus.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# WHY chunk: the corpus is one long file; a training example needs to fit MAX_SEQ_LENGTH.
tokens = tokenizer(raw_text, return_tensors=None)["input_ids"]
chunk_size = MAX_SEQ_LENGTH
chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size) if len(tokens[i:i + chunk_size]) > 20]
pretrain_texts = [tokenizer.decode(c) for c in chunks]

pretrain_dataset = Dataset.from_dict({"text": pretrain_texts})
print(f"Continued-pretraining chunks: {len(pretrain_dataset)}")


In [ ]:
# WHAT: a short continued-pretraining run (few epochs, small dataset -> keep it brief).
# WHY these training-argument choices:
#   - per_device_train_batch_size=2, gradient_accumulation_steps=4:
#       effective batch size = 2*4 = 8. We keep the *real* batch size small because a
#       T4 has limited VRAM, and use gradient accumulation to simulate a larger batch
#       (more stable gradients) without running out of memory.
#   - learning_rate=1e-4: continued pretraining on a small domain corpus uses a slightly
#       higher LR than typical full pretraining (which uses ~1e-5..5e-5) because we are
#       only updating a small LoRA adapter, not the full network.
#   - num_train_epochs=1: continued pretraining on a small corpus is meant to be a light
#       nudge, not exhaustive training - 1 pass avoids overfitting to a small raw-text file.
pretrain_args = SFTConfig(
    output_dir="outputs/stage1_continued_pretraining",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=1e-4,
    logging_steps=5,
    optim="adamw_8bit",     # WHY 8-bit optimizer: halves optimizer-state memory vs fp32 Adam.
    warmup_steps=10,        # WHY warmup: avoids a large, unstable gradient step right at the start.
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

pretrain_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=pretrain_dataset,
    args=pretrain_args,
)

pretrain_trainer.train()


## Step 5 — Load the SFT dataset

**What:** load `instruction_dataset.jsonl` and format each `{instruction, response}` pair into the model's chat template.
**Why:** the base model expects a specific chat format (system/user/assistant turns) - training on correctly-templated text is what lets the model respond well when *you* later chat with it in the same format.
**How:** Unsloth's tokenizer exposes `apply_chat_template`, matching the format the base checkpoint was originally instruct-tuned with.

In [ ]:
# WHAT: load the instruction/response pairs for SFT.
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

instruction_rows = load_jsonl("data/instruction_dataset.jsonl")
print(f"Loaded {len(instruction_rows)} instruction-response pairs")
print(instruction_rows[0])


In [ ]:
# WHAT: convert each {instruction, response} pair into the model's chat template.
# WHY: SFTTrainer trains on plain text, so we must pre-render the chat structure
#      (who said what) into a single string per example, matching inference-time format.
def format_example(example):
    messages = [
        {"role": "system", "content": "You are PostTraining Tutor, an assistant that explains LLM training concepts (Transformers, LoRA, QLoRA, SFT, DPO, RLHF, alignment) clearly and concisely."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

sft_dataset = Dataset.from_list(instruction_rows).map(format_example)
print(sft_dataset[0]["text"][:500])


## Step 6 — Training arguments + SFTTrainer (Stage 2)

**Key parameters, explained again in the SFT context (values may reasonably differ from Stage 1):**
- `num_train_epochs=3`: SFT datasets are usually small and hand-curated (unlike raw pretraining text), so multiple passes help the model reliably learn the instruction->response *pattern* without needing a huge dataset.
- `per_device_train_batch_size=2` / `gradient_accumulation_steps=4`: same reasoning as Stage 1 - keep real memory use low, simulate a larger effective batch (8) for gradient stability.
- `learning_rate=2e-4`: a common LoRA SFT learning rate - LoRA adapters (small parameter count) tolerate a higher LR than full fine-tuning would.
- `optim="adamw_8bit"`: 8-bit AdamW keeps optimizer state memory low, important with a 4-bit base model already using most of the T4's VRAM budget.


In [ ]:
# WHAT: define SFT training arguments and launch the trainer.
sft_args = SFTConfig(
    output_dir="outputs/stage2_sft",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    optim="adamw_8bit",
    warmup_steps=10,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    args=sft_args,
)

sft_trainer.train()


## Step 7 — Save (and optionally upload) the SFT adapter

**What:** save only the LoRA adapter weights (a few MB), not the full 3B-parameter model.
**Why:** this is the entire point of LoRA/QLoRA - the adapter is small, portable, and can be re-applied on top of the same base model at inference time or before Stage 3 (DPO).
**How:** `save_pretrained` on a PEFT-wrapped model saves only the adapter by default.

In [ ]:
# WHAT: save the LoRA adapter locally.
SFT_ADAPTER_DIR = "outputs/sft_adapter"
model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print(f"Saved SFT adapter to {SFT_ADAPTER_DIR}")


In [ ]:
# WHAT (optional): push the adapter to the Hugging Face Hub so Notebook 2 can load it
#      from anywhere, and so it's shareable in your talk.
# WHY: keeps the pipeline reproducible without re-uploading files by hand between notebooks.
# HOW: requires `huggingface_hub` login - uncomment and set your repo id/token to use.

# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("your-username/postraining-tutor-sft-adapter")
# tokenizer.push_to_hub("your-username/postraining-tutor-sft-adapter")


## Step 8 — Inference: before vs. after SFT

**What:** ask the same question to the model before and after this notebook's training.
**Why:** a quick, visible sanity check that training actually changed behavior in the intended direction, before moving to Notebook 2.
**How:** `FastLanguageModel.for_inference` applies Unsloth's inference-speed optimizations (e.g. native 2x faster generation).

In [ ]:
# WHAT: run a couple of test prompts through the freshly SFT-trained model.
FastLanguageModel.for_inference(model)  # WHY: switches Unsloth into its faster inference mode.

test_questions = [
    "What is LoRA and why is it useful for fine-tuning large language models?",
    "Explain the difference between SFT and DPO in simple terms.",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "You are PostTraining Tutor, an assistant that explains LLM training concepts clearly and concisely."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    output = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {response}\n{'-'*80}")


**Next:** open `02_DPO_Alignment.ipynb` and load `outputs/sft_adapter` (or your uploaded Hub repo) to run Stage 3.